<h3>PDF의 텍스트 층과 표</h3>
<p><strong>개념</strong> PDF는 화면 배치를 저장하는 형식입니다. 글자가 선택되는 디지털 PDF도 읽기 순서와 표의 행·열 관계가 보존되는지 따로 확인해야 합니다.</p>

In [1]:
# %pip은 현재 노트북 커널에 PDF 읽기 패키지를 설치합니다. -q는 설치 로그를 줄입니다.
# 버전을 고정해 학생 환경에서도 같은 API를 사용합니다.
%pip install -q "pypdf==6.18.0" "pdfplumber==0.11.10"


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Path.cwd()와 parents로 현재 위치부터 상위 폴더를 검사해 저장소 루트를 찾습니다.
# pdf_path는 텍스트 PDF, scan_path는 텍스트 층이 없는 스캔 PDF를 가리킵니다.
from pathlib import Path
from pypdf import PdfReader
import pdfplumber

roots = [Path.cwd(), *Path.cwd().parents]
repos = [p for p in roots if (p / "labs/data/rag-corpus").is_dir()]
if not repos: raise FileNotFoundError("저장소 루트에서 노트북을 실행하세요")
repo = repos[0]
pdf_path = repo / "labs/data/rag-corpus/pdf/easy/E02.pdf"
scan_path = repo / "labs/data/rag-corpus/pdf/scanned/S01-E01-scan.pdf"


<h3>기본 추출과 표 추출 비교</h3>
<p><strong>확인할 점</strong> 같은 쪽을 두 파서로 읽고 글자 수와 표 개수를 함께 봅니다. 수치가 크다는 사실만으로 구조가 정확하다고 판단하지 않습니다.</p>

In [3]:
# PdfReader는 쪽별 일반 텍스트를, pdfplumber는 레이아웃 텍스트와 표를 반환합니다.
# pages[1]은 0부터 세므로 실제 두 번째 쪽이며, or ""는 None을 빈 문자열로 바꿉니다.
reader = PdfReader(pdf_path)
plain = reader.pages[1].extract_text() or ""
with pdfplumber.open(pdf_path) as pdf:
    layout = pdf.pages[1].extract_text(layout=True) or ""
    tables = pdf.pages[1].extract_tables()
print({"pages": len(reader.pages), "plain_chars": len(plain),
       "layout_chars": len(layout), "tables_on_page_2": len(tables)})


{'pages': 5, 'plain_chars': 884, 'layout_chars': 5311, 'tables_on_page_2': 1}


<h3>일부러 실패해 보기</h3>
<p><strong>실패 사례</strong> 스캔 PDF에서 텍스트가 바로 추출된다고 가정하면 빈 문자열을 정상 본문으로 처리합니다.</p>

In [4]:
# 스캔 PDF도 바로 텍스트가 나온다고 가정해 실패를 재현합니다.
# all()은 모든 쪽에 글자가 있을 때만 True이며 AssertionError만 포착합니다.
try:
    scan = PdfReader(scan_path)
    extracted = [page.extract_text() or "" for page in scan.pages]
    assert all(text.strip() for text in extracted), "텍스트 층이 없는 쪽이 있습니다"
except AssertionError as error:
    print(f"caught: {error}")


caught: 텍스트 층이 없는 쪽이 있습니다


<p><strong>수정 이유</strong> 각 쪽의 텍스트 길이를 확인하고 비어 있는 쪽만 OCR 후보로 분류합니다. 이 셀은 OCR을 실행하지 않습니다.</p>

In [5]:
# 파일 존재를 먼저 확인한 뒤, 추출 문자열이 20자 미만인 쪽을 OCR 후보로 고릅니다.
# enumerate(..., 1)은 쪽 번호를 학생이 보는 1부터 시작하게 합니다.
if not scan_path.is_file(): raise FileNotFoundError(scan_path)
scan = PdfReader(scan_path)
missing = [i for i, page in enumerate(scan.pages, 1)
           if len((page.extract_text() or "").strip()) < 20]
assert missing == list(range(1, len(scan.pages) + 1))
print("OCR candidate pages:", missing)


OCR candidate pages: [1, 2, 3, 4, 5]
